# SHA-256 Transfer Grammar v2

## Identity-Filtered, Domain-Conditioned, Classifier-Audited

This notebook continues the external rail:

$$
\boxed{
\tau(M)\rightarrow \Delta B(H)
}
$$

It fixes the main weakness in v1: some transformations are **identity-equivalent** on flat domains.

Example:

$$
\operatorname{reverse}(00\ldots00)=00\ldots00
$$

So v2 reports two tracks:

$$
\boxed{\text{all samples}}
$$

and

$$
\boxed{\text{active samples only: }\tau(M)\ne M.}
$$

Core object:

$$
\Delta B_H(\tau;M)=B(H(\tau(M)))-B(H(M)).
$$

Scope guard:

$$
\boxed{
\text{This notebook measures input-shape transfer. It does not claim SHA inversion.}
}
$$


In [ ]:
# Top cell: imports and settings

import hashlib
import math
import random
from typing import Callable, Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260501 + 2
random.seed(SEED)
np.random.seed(SEED)

N_BASE = 128
MSG_LEN = 64
AUTOCORR_LAGS = 16
TEST_FRAC = 0.25

print("SHA Transfer Grammar v2 initialized.")
print(f"SEED={SEED}, N_BASE={N_BASE}, MSG_LEN={MSG_LEN}, AUTOCORR_LAGS={AUTOCORR_LAGS}")


## 1. Digest behavior vector $B(H)$

The output behavior vector is deliberately external:

- byte entropy,
- byte mean/std,
- bit density,
- autocorrelation lags $1..16$,
- autocorrelation sign changes,
- crude typeless byte-stream probes.

These are not internal SHA states. They are output-face measurements.


In [ ]:
# Digest behavior features

def sha256_bytes(msg: bytes) -> bytes:
    return hashlib.sha256(msg).digest()

def byte_entropy(bs: bytes) -> float:
    counts = np.bincount(np.frombuffer(bs, dtype=np.uint8), minlength=256)
    p = counts[counts > 0] / len(bs)
    return float(-(p * np.log2(p)).sum())

def digest_autocorr(bs: bytes, max_lag: int = AUTOCORR_LAGS) -> np.ndarray:
    x = np.frombuffer(bs, dtype=np.uint8).astype(float)
    x = x - x.mean()
    denom = float(np.dot(x, x))
    if denom == 0:
        return np.zeros(max_lag)
    return np.array([float(np.dot(x[:-lag], x[lag:]) / denom) for lag in range(1, max_lag + 1)])

def sign_changes(v: np.ndarray) -> int:
    s = np.sign(v)
    nz = s[s != 0]
    if len(nz) < 2:
        return 0
    return int(np.sum(nz[:-1] * nz[1:] < 0))

def bit_density(bs: bytes) -> float:
    return sum(int(b).bit_count() for b in bs) / (8 * len(bs))

def byte_class_features(bs: bytes) -> Dict[str, float]:
    arr = list(bs)
    n = max(1, len(arr))
    ctrl = sum((0x70 <= b <= 0x7F) or b in {0xE8,0xE9,0xEB,0xC2,0xC3,0xCA,0xCB,0xCC} for b in arr) / n
    stack = sum(b in {0x50,0x51,0x52,0x53,0x54,0x55,0x56,0x57,0x58,0x59,0x5A,0x5B,0x5C,0x5D,0x5E,0x5F} for b in arr) / n
    printable = sum(32 <= b <= 126 for b in arr) / n
    flat = sum(b in {0x00,0x90,0xFF} for b in arr) / n
    repeats = sum(arr[i] == arr[i-1] for i in range(1, len(arr))) / max(1, len(arr)-1)
    high_nibble_repeat = sum((arr[i] >> 4) == (arr[i-1] >> 4) for i in range(1, len(arr))) / max(1, len(arr)-1)
    low_nibble_repeat = sum((arr[i] & 15) == (arr[i-1] & 15) for i in range(1, len(arr))) / max(1, len(arr)-1)
    return {
        "x86_ctrl_proxy": float(ctrl),
        "x86_stack_proxy": float(stack),
        "ascii_printable_proxy": float(printable),
        "zero_ff_nop_proxy": float(flat),
        "byte_repeat_proxy": float(repeats),
        "high_nibble_repeat_proxy": float(high_nibble_repeat),
        "low_nibble_repeat_proxy": float(low_nibble_repeat),
    }

def B_of_digest(digest: bytes) -> Dict[str, float]:
    arr = np.frombuffer(digest, dtype=np.uint8).astype(float)
    ac = digest_autocorr(digest, AUTOCORR_LAGS)
    feats = {
        "entropy": byte_entropy(digest),
        "byte_mean": float(arr.mean()),
        "byte_std": float(arr.std()),
        "bit_density": bit_density(digest),
        "ac_sign_changes": float(sign_changes(ac)),
        "ac_energy": float(np.dot(ac, ac)),
        "ac_abs_mean": float(np.mean(np.abs(ac))),
        "ac_max": float(np.max(ac)),
        "ac_min": float(np.min(ac)),
        "lag1": float(ac[0]),
        "lag2": float(ac[1]),
        "lag4": float(ac[3]),
        "lag8": float(ac[7]),
        "lag16": float(ac[15]),
    }
    for i, val in enumerate(ac, start=1):
        feats[f"ac_{i:02d}"] = float(val)
    feats.update(byte_class_features(digest))
    return feats

def B_of_msg(msg: bytes) -> Dict[str, float]:
    return B_of_digest(sha256_bytes(msg))

sample = b"abc"
feats = B_of_msg(sample)
print("Feature count:", len(feats))
print("Sample digest:", sha256_bytes(sample).hex())
print({k: round(feats[k], 4) for k in ["entropy", "bit_density", "ac_sign_changes", "lag1"]})


## 2. Source measures $\mu_D$

The source measures are intentionally simple. They are **shape classes**, not final ontology.

All messages are fixed length so length cannot dominate the readout.


In [ ]:
# Source generators

WORDS = (
    "the quick brown fox jumps over lazy dog nexus phase fold shape signal "
    "cipher lattice matrix carry scar grammar recursion input output transfer "
    "field boundary echo attractor wave stack layer proof "
).split()

CODE_TOKENS = [
    b"push", b"mov", b"xor", b"add", b"sub", b"jmp", b"call", b"ret",
    b"cmp", b"jne", b"loop", b"int", b"nop", b"lea", b"shr", b"ror",
    b"and", b"or", b"test", b"inc", b"dec"
]

def fix_len(bs: bytes, n: int = MSG_LEN) -> bytes:
    if len(bs) >= n:
        return bs[:n]
    return bs + bytes([0]) * (n - len(bs))

def gen_random(n=MSG_LEN): 
    return bytes(random.getrandbits(8) for _ in range(n))

def gen_zero(n=MSG_LEN):
    return bytes([0]) * n

def gen_nop(n=MSG_LEN):
    return bytes([0x90]) * n

def gen_ff(n=MSG_LEN):
    return bytes([0xFF]) * n

def gen_repeated_byte(n=MSG_LEN):
    b = random.randrange(256)
    return bytes([b]) * n

def gen_repeated_word(n=MSG_LEN):
    w = random.getrandbits(32).to_bytes(4, "big")
    return (w * ((n+3)//4))[:n]

def gen_two_word_repeat(n=MSG_LEN):
    w = random.getrandbits(64).to_bytes(8, "big")
    return (w * ((n+7)//8))[:n]

def gen_text(n=MSG_LEN):
    out = []
    while len(" ".join(out).encode()) < n:
        out.append(random.choice(WORDS))
    return fix_len(" ".join(out).encode(), n)

def gen_code_like(n=MSG_LEN):
    out = bytearray()
    while len(out) < n:
        out.extend(random.choice(CODE_TOKENS))
        out.append(random.choice([0x20,0x0A,0x3B,0x00,0x90]))
    return bytes(out[:n])

def gen_tone(n=MSG_LEN):
    phase = random.random() * 2 * math.pi
    freq = random.choice([1,2,3,5,8,13])
    vals = []
    for i in range(n):
        v = 128 + 80 * math.sin(2 * math.pi * freq * i / n + phase)
        vals.append(int(max(0, min(255, round(v)))))
    return bytes(vals)

GENERATORS = {
    "random": gen_random,
    "zero": gen_zero,
    "nop": gen_nop,
    "ff": gen_ff,
    "repeated_byte": gen_repeated_byte,
    "repeated_word": gen_repeated_word,
    "two_word_repeat": gen_two_word_repeat,
    "text": gen_text,
    "code_like": gen_code_like,
    "tone": gen_tone,
}

base_rows = []
base_messages = []
for domain, gen in GENERATORS.items():
    for idx in range(N_BASE):
        msg = gen(MSG_LEN)
        base_messages.append((domain, idx, msg))
        row = {"domain": domain, "idx": idx, "msg_hex": msg.hex()}
        row.update(B_of_msg(msg))
        base_rows.append(row)

base_df = pd.DataFrame(base_rows)
feature_cols = [c for c in base_df.columns if c not in {"domain", "idx", "msg_hex"}]

print("Base dataset:", base_df.shape)
base_df.groupby("domain")[["entropy", "ac_sign_changes", "ac_energy", "x86_ctrl_proxy"]].mean().round(4)


## 3. Input verbs $\tau$

v2 preserves the prior transforms and adds a few targeted “shape verbs”.

The notebook logs:

$$
\mathbf{1}[\tau(M)=M]
$$

so identity-equivalent cases can be removed.


In [ ]:
# Input transformations

def tau_identity(m: bytes) -> bytes:
    return bytes(m)

def tau_reverse(m: bytes) -> bytes:
    return bytes(reversed(m))

def tau_rotate_left_1(m: bytes) -> bytes:
    return m[1:] + m[:1] if m else m

def tau_rotate_left_4(m: bytes) -> bytes:
    return m[4:] + m[:4] if len(m) >= 4 else m

def tau_rotate_half(m: bytes) -> bytes:
    k = len(m)//2
    return m[k:] + m[:k]

def tau_word_permute(m: bytes) -> bytes:
    chunks = [m[i:i+4] for i in range(0, len(m), 4)]
    perm = chunks[::2] + chunks[1::2]
    return b"".join(perm)[:len(m)]

def tau_bit_not(m: bytes) -> bytes:
    return bytes((~b) & 0xFF for b in m)

def tau_xor_55(m: bytes) -> bytes:
    return bytes(b ^ 0x55 for b in m)

def tau_xor_aa(m: bytes) -> bytes:
    return bytes(b ^ 0xAA for b in m)

def tau_flip_first_bit(m: bytes) -> bytes:
    out = bytearray(m)
    if out:
        out[0] ^= 0x80
    return bytes(out)

def tau_flip_last_bit(m: bytes) -> bytes:
    out = bytearray(m)
    if out:
        out[-1] ^= 0x01
    return bytes(out)

def tau_flip_middle_bit(m: bytes) -> bytes:
    out = bytearray(m)
    if out:
        out[len(out)//2] ^= 0x08
    return bytes(out)

def tau_repeat_first_byte(m: bytes) -> bytes:
    return bytes([m[0] if m else 0]) * len(m)

def tau_repeat_first_word(m: bytes) -> bytes:
    w = (m[:4] + b"\x00\x00\x00\x00")[:4]
    return (w * ((len(m)+3)//4))[:len(m)]

def tau_repeat_first_8(m: bytes) -> bytes:
    w = (m[:8] + b"\x00"*8)[:8]
    return (w * ((len(m)+7)//8))[:len(m)]

def tau_zero_second_half(m: bytes) -> bytes:
    k = len(m)//2
    return m[:k] + bytes(len(m)-k)

def tau_randomize_second_half(m: bytes) -> bytes:
    k = len(m)//2
    return m[:k] + bytes(random.getrandbits(8) for _ in range(len(m)-k))

def tau_sort_bytes(m: bytes) -> bytes:
    return bytes(sorted(m))

def tau_ascii_clamp(m: bytes) -> bytes:
    return bytes(32 + (b % 95) for b in m)

def tau_codeify(m: bytes) -> bytes:
    vocab = [0x90,0x55,0x8B,0xE8,0xC3,0x74,0x75,0x31,0xC0,0x50,0x58,0xEB]
    return bytes(vocab[b % len(vocab)] for b in m)

def tau_low_nibble_keep(m: bytes) -> bytes:
    return bytes(b & 0x0F for b in m)

def tau_high_nibble_keep(m: bytes) -> bytes:
    return bytes(b & 0xF0 for b in m)

TRANSFORMS = {
    "identity": tau_identity,
    "reverse": tau_reverse,
    "rotate_left_1": tau_rotate_left_1,
    "rotate_left_4": tau_rotate_left_4,
    "rotate_half": tau_rotate_half,
    "word_permute": tau_word_permute,
    "bit_not": tau_bit_not,
    "xor_55": tau_xor_55,
    "xor_aa": tau_xor_aa,
    "flip_first_bit": tau_flip_first_bit,
    "flip_middle_bit": tau_flip_middle_bit,
    "flip_last_bit": tau_flip_last_bit,
    "repeat_first_byte": tau_repeat_first_byte,
    "repeat_first_word": tau_repeat_first_word,
    "repeat_first_8": tau_repeat_first_8,
    "zero_second_half": tau_zero_second_half,
    "randomize_second_half": tau_randomize_second_half,
    "sort_bytes": tau_sort_bytes,
    "ascii_clamp": tau_ascii_clamp,
    "codeify": tau_codeify,
    "low_nibble_keep": tau_low_nibble_keep,
    "high_nibble_keep": tau_high_nibble_keep,
}

print("Transform count:", len(TRANSFORMS))
print(list(TRANSFORMS))


## 4. Build all-sample and active-only transfer tables

All-sample table:

$$
\Delta B_H(\tau;M)
$$

Active-only table:

$$
\Delta B_H(\tau;M)\quad\text{where}\quad \tau(M)\ne M.
$$


In [ ]:
# Transfer table construction

rows = []
for domain, idx, msg in base_messages:
    B0 = B_of_msg(msg)
    for tname, tfun in TRANSFORMS.items():
        msg2 = tfun(msg)
        B1 = B_of_msg(msg2)
        row = {
            "domain": domain,
            "idx": idx,
            "transform": tname,
            "same_input": msg == msg2,
            "input_hamming": sum((a ^ b).bit_count() for a, b in zip(msg, msg2)),
        }
        for c in feature_cols:
            row[f"d_{c}"] = B1[c] - B0[c]
        rows.append(row)

transfer_df = pd.DataFrame(rows)
delta_cols = [c for c in transfer_df.columns if c.startswith("d_")]
active_df = transfer_df[~transfer_df["same_input"]].copy()

print("All transfer rows:", transfer_df.shape)
print("Active-only rows:", active_df.shape)
print("Identity-equivalent rate:", round(float(transfer_df["same_input"].mean()), 4))

same_rate = transfer_df.groupby("transform")["same_input"].mean().sort_values(ascending=False)
same_rate.round(4)


## 5. Grammar summaries: all vs active-only

This is the core v2 correction.

A transform can appear weak in the all-sample view if many samples are identity-equivalent.


In [ ]:
# Grammar summary helper

def grammar_summary(df: pd.DataFrame) -> pd.DataFrame:
    agg = df.groupby("transform")[delta_cols].mean()
    norm = np.sqrt((agg ** 2).sum(axis=1))
    out = pd.DataFrame({
        "mean_delta_norm": norm,
        "mean_input_hamming": df.groupby("transform")["input_hamming"].mean(),
        "n_samples": df.groupby("transform").size(),
        "same_input_rate_in_source": transfer_df.groupby("transform")["same_input"].mean(),
    }).sort_values("mean_delta_norm", ascending=False)
    return out

summary_all = grammar_summary(transfer_df)
summary_active = grammar_summary(active_df)

print("All-sample top transforms")
display(summary_all.round(4).head(15))
print("Active-only top transforms")
display(summary_active.round(4).head(15))


In [ ]:
# Compare all vs active-only transform strengths

compare = summary_all[["mean_delta_norm"]].rename(columns={"mean_delta_norm": "all"})
compare["active_only"] = summary_active["mean_delta_norm"]
compare["active_minus_all"] = compare["active_only"] - compare["all"]
compare = compare.sort_values("active_minus_all", ascending=False)

display(compare.round(4).head(20))

plt.figure(figsize=(12, 6))
plot_df = compare.sort_values("active_only")
plt.plot(plot_df["all"].values, range(len(plot_df)), marker="o", label="all")
plt.plot(plot_df["active_only"].values, range(len(plot_df)), marker="x", label="active only")
plt.yticks(range(len(plot_df)), plot_df.index)
plt.xlabel(r"$||E[\Delta B_H|\tau]||_2$")
plt.title("Transform strength: all samples vs active-only")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Heatmaps: all vs active-only

The transfer grammar sheet should be read in both modes.


In [ ]:
# Heatmap function

compact_delta_cols = [
    "d_entropy", "d_byte_std", "d_bit_density", "d_ac_sign_changes",
    "d_ac_energy", "d_ac_abs_mean", "d_ac_max", "d_ac_min",
    "d_lag1", "d_lag2", "d_lag4", "d_lag8",
    "d_x86_ctrl_proxy", "d_x86_stack_proxy", "d_ascii_printable_proxy",
    "d_zero_ff_nop_proxy", "d_byte_repeat_proxy",
    "d_high_nibble_repeat_proxy", "d_low_nibble_repeat_proxy",
]

def plot_heatmap(df: pd.DataFrame, summary: pd.DataFrame, title: str):
    agg = df.groupby("transform")[compact_delta_cols].mean()
    agg = agg.loc[summary.index]
    z = (agg - agg.mean(axis=0)) / (agg.std(axis=0).replace(0, 1))
    plt.figure(figsize=(13, 8))
    plt.imshow(z.values, aspect="auto")
    plt.colorbar(label="column-normalized mean delta")
    plt.yticks(range(len(z.index)), z.index)
    plt.xticks(range(len(z.columns)), [c.replace("d_", "") for c in z.columns], rotation=90)
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_heatmap(transfer_df, summary_all, "Transfer grammar heatmap: all samples")
plot_heatmap(active_df, summary_active, "Transfer grammar heatmap: active-only samples")


## 7. Domain-conditioned active grammar sheets

Now condition on source measure:

$$
\|\mathbb E[\Delta B_H\mid \mu_D,\tau,\tau(M)\ne M]\|_2.
$$


In [ ]:
# Domain-conditioned active grammar

domain_transform_active = (
    active_df
    .groupby(["domain", "transform"])[delta_cols]
    .mean()
    .apply(lambda row: float(np.sqrt(np.dot(row, row))), axis=1)
    .unstack("transform")
    .fillna(0.0)
)

display(domain_transform_active.round(4))

plt.figure(figsize=(14, 7))
plt.imshow(domain_transform_active.values, aspect="auto")
plt.colorbar(label=r"$||E[\Delta B_H|\mu_D,\tau,active]||_2$")
plt.yticks(range(len(domain_transform_active.index)), domain_transform_active.index)
plt.xticks(range(len(domain_transform_active.columns)), domain_transform_active.columns, rotation=90)
plt.title("Domain-conditioned transfer strength, active-only")
plt.tight_layout()
plt.show()


## 8. Classifier audit with Wilson confidence intervals

We test whether the output deformation can identify the input verb:

$$
\Delta B_H\rightarrow \hat{\tau}
$$

This is not expected to be perfect. The meaningful question is whether it is above random baseline after identity-equivalent cases are removed.


In [ ]:
# Classifier helpers

def wilson_interval(successes: int, n: int, z: float = 1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    phat = successes / n
    denom = 1 + z*z/n
    center = (phat + z*z/(2*n)) / denom
    half = z * math.sqrt((phat*(1-phat) + z*z/(4*n)) / n) / denom
    return center - half, center + half

def split_by_group(df: pd.DataFrame, test_frac: float, group_cols: List[str]):
    train_parts, test_parts = [], []
    for _, g in df.groupby(group_cols):
        idxs = list(g.index)
        random.shuffle(idxs)
        n_test = max(1, int(round(len(idxs) * test_frac)))
        test_parts.append(df.loc[idxs[:n_test]])
        train_parts.append(df.loc[idxs[n_test:]])
    return pd.concat(train_parts), pd.concat(test_parts)

def nearest_centroid_eval(df: pd.DataFrame, label_col: str, cols: List[str], group_cols: List[str]):
    train, test = split_by_group(df, TEST_FRAC, group_cols)
    mu = train[cols].mean()
    sd = train[cols].std().replace(0, 1)
    Z_train = (train[cols] - mu) / sd
    Z_test = (test[cols] - mu) / sd
    centroids = Z_train.join(train[label_col]).groupby(label_col).mean()

    preds = []
    C = centroids.values
    labels = list(centroids.index)
    for i in range(len(Z_test)):
        z = Z_test.iloc[i].values
        d = np.sum((C - z)**2, axis=1)
        preds.append(labels[int(np.argmin(d))])
    truth = list(test[label_col])
    correct = sum(p == t for p, t in zip(preds, truth))
    acc = correct / len(test)
    lo, hi = wilson_interval(correct, len(test))
    eval_df = test.copy()
    eval_df[f"pred_{label_col}"] = preds
    return acc, lo, hi, eval_df

# Remove identity transform and identity-equivalent samples for transform classifier.
active_for_transform = active_df[active_df["transform"] != "identity"].copy()
acc_t, lo_t, hi_t, eval_t = nearest_centroid_eval(
    active_for_transform, 
    "transform", 
    delta_cols, 
    ["domain", "transform"]
)

n_classes_t = active_for_transform["transform"].nunique()
print(f"Active-only transform recognition accuracy: {acc_t:.3f}")
print(f"Wilson 95% CI: [{lo_t:.3f}, {hi_t:.3f}]")
print(f"Random baseline: {1/n_classes_t:.3f}")
print("Classes:", n_classes_t)

conf_t = pd.crosstab(eval_t["transform"], eval_t["pred_transform"], normalize="index").round(2)
conf_t


## 9. Domain classifier audit

Test:

$$
B(H)\rightarrow \hat{\mu}_D
$$

This checks whether digest behavior carries source-measure signal.


In [ ]:
# Domain recognition with Wilson interval

acc_d, lo_d, hi_d, eval_d = nearest_centroid_eval(
    base_df,
    "domain",
    feature_cols,
    ["domain"]
)

n_classes_d = base_df["domain"].nunique()
print(f"Domain recognition accuracy from B(H): {acc_d:.3f}")
print(f"Wilson 95% CI: [{lo_d:.3f}, {hi_d:.3f}]")
print(f"Random baseline: {1/n_classes_d:.3f}")
print("Classes:", n_classes_d)

pd.crosstab(eval_d["domain"], eval_d["pred_domain"], normalize="index").round(2)


## 10. Stronger model option if scikit-learn is available

Nearest-centroid is deliberately crude. If `sklearn` exists, this cell runs a logistic classifier with standardized features.

If not installed, it safely skips.


In [ ]:
# Optional sklearn audit

try:
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, confusion_matrix

    tr, te = split_by_group(active_for_transform, TEST_FRAC, ["domain", "transform"])
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None)
    )
    clf.fit(tr[delta_cols].values, tr["transform"].values)
    pred = clf.predict(te[delta_cols].values)
    acc = accuracy_score(te["transform"].values, pred)
    lo, hi = wilson_interval(int((pred == te["transform"].values).sum()), len(te))
    print(f"Logistic transform classifier accuracy: {acc:.3f}")
    print(f"Wilson 95% CI: [{lo:.3f}, {hi:.3f}]")
    print(f"Random baseline: {1/active_for_transform['transform'].nunique():.3f}")

    trd, ted = split_by_group(base_df, TEST_FRAC, ["domain"])
    clf_d = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=None)
    )
    clf_d.fit(trd[feature_cols].values, trd["domain"].values)
    pred_d = clf_d.predict(ted[feature_cols].values)
    accd = accuracy_score(ted["domain"].values, pred_d)
    lod, hid = wilson_interval(int((pred_d == ted["domain"].values).sum()), len(ted))
    print()
    print(f"Logistic domain classifier accuracy: {accd:.3f}")
    print(f"Wilson 95% CI: [{lod:.3f}, {hid:.3f}]")
    print(f"Random baseline: {1/base_df['domain'].nunique():.3f}")
except Exception as e:
    print("sklearn classifier skipped.")
    print("Reason:", repr(e))


## 11. One-bit avalanche, active and domain-conditioned

Avalanche remains globally centered near $128$ digest bits, but behavior-space deformation is domain-conditioned.


In [ ]:
# One-bit avalanche experiment

def flip_bit(m: bytes, bit_index: int) -> bytes:
    out = bytearray(m)
    byte_i = bit_index // 8
    bit_i = 7 - (bit_index % 8)
    out[byte_i] ^= (1 << bit_i)
    return bytes(out)

av_rows = []
for domain, idx, msg in base_messages:
    bit_index = random.randrange(len(msg) * 8)
    msg2 = flip_bit(msg, bit_index)
    h0 = sha256_bytes(msg)
    h1 = sha256_bytes(msg2)
    digest_hd = sum((a ^ b).bit_count() for a, b in zip(h0, h1))
    B0 = B_of_digest(h0)
    B1 = B_of_digest(h1)
    row = {"domain": domain, "idx": idx, "bit_index": bit_index, "digest_hamming": digest_hd}
    for c in feature_cols:
        row[f"d_{c}"] = B1[c] - B0[c]
    av_rows.append(row)

av_df = pd.DataFrame(av_rows)
av_delta_cols = [c for c in av_df.columns if c.startswith("d_")]
av_df["delta_norm"] = np.sqrt((av_df[av_delta_cols] ** 2).sum(axis=1))

print("Digest avalanche Hamming summary")
display(av_df["digest_hamming"].describe().round(3))
print("Behavior deformation by domain")
display(av_df.groupby("domain")["delta_norm"].describe().round(3))

plt.figure(figsize=(8, 4))
plt.hist(av_df["digest_hamming"], bins=24)
plt.title("Digest Hamming distance after one-bit input flip")
plt.xlabel("digest Hamming distance")
plt.ylabel("count")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
av_df.groupby("domain")["delta_norm"].mean().sort_values().plot(kind="barh")
plt.title("Mean behavior-space deformation from one-bit input flip")
plt.xlabel(r"$||\Delta B_H||_2$")
plt.ylabel("domain")
plt.tight_layout()
plt.show()


## 12. PCA transfer map, active-only

This is a visualization of the active transfer surface.

Low separability here is expected: SHA deliberately mixes local moves into global-looking output. The question is not whether the clusters are perfectly separated, but whether the echoes are measurably nonzero and domain-conditioned.


In [ ]:
# PCA via SVD on active deltas

X = active_for_transform[delta_cols].values.astype(float)
X_mean = X.mean(axis=0, keepdims=True)
X_std = X.std(axis=0, keepdims=True)
X_std[X_std == 0] = 1
Z = (X - X_mean) / X_std

U, S, Vt = np.linalg.svd(Z, full_matrices=False)
coords = U[:, :2] * S[:2]
active_for_transform = active_for_transform.copy()
active_for_transform["pc1"] = coords[:, 0]
active_for_transform["pc2"] = coords[:, 1]

explained = (S**2) / np.sum(S**2)
print("Explained variance first 5 PCs:", np.round(explained[:5], 4))

plt.figure(figsize=(9, 7))
for tname in sorted(active_for_transform["transform"].unique()):
    sub = active_for_transform[active_for_transform["transform"] == tname]
    plt.scatter(sub["pc1"], sub["pc2"], s=10, alpha=0.45, label=tname)
plt.title("Active transfer deltas in output-behavior PCA space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()


## 13. Export v2 artifacts

This writes:

1. `sha_transfer_v2_base_features.csv`
2. `sha_transfer_v2_all_delta_grammar.csv`
3. `sha_transfer_v2_active_delta_grammar.csv`
4. `sha_transfer_v2_summary_all.csv`
5. `sha_transfer_v2_summary_active.csv`
6. `sha_transfer_v2_domain_active_norm.csv`


In [ ]:
# Export artifacts

base_df.to_csv("sha_transfer_v2_base_features.csv", index=False)
transfer_df.to_csv("sha_transfer_v2_all_delta_grammar.csv", index=False)
active_df.to_csv("sha_transfer_v2_active_delta_grammar.csv", index=False)
summary_all.to_csv("sha_transfer_v2_summary_all.csv")
summary_active.to_csv("sha_transfer_v2_summary_active.csv")
domain_transform_active.to_csv("sha_transfer_v2_domain_active_norm.csv")

print("Exported v2 CSV artifacts.")


# Final Ψ-collapse

v2 corrects the prior blur:

$$
\boxed{
\tau(M)=M\ \text{cases must be separated from active transformations.}
}
$$

The notebook now gives:

$$
\boxed{
\text{all-sample grammar}
}
$$

and:

$$
\boxed{
\text{active-only grammar}
}
$$

plus Wilson intervals and classifier audits.

The expected pattern is:

$$
\boxed{
B(H)\text{ carries stronger source-domain signal than exact transform signal.}
}
$$

That supports the stable Nexus reading:

$$
\boxed{
\text{digest behavior selects manifolds/templates better than it recovers parameters.}
}
$$

Next notebook rail:

$$
\boxed{
\text{Transfer Grammar v3: controlled families and monotone ramps}
}
$$

where each input source is moved smoothly along a parameter $\lambda$:

$$
M_\lambda=(1-\lambda)M_{\text{structured}}\oplus \lambda M_{\text{random}}
$$

and we test whether $B(H)$ follows a monotone / curved path in behavior space.
